## RAG 모델 성능 평가하기

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
file_path = "../../data/Sustainability_report_2024_kr.pdf"

loader = PyPDFLoader(file_path)

docs = loader.load()
len(docs)
docs[:3]

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 0, 'page_label': '1'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 1, 'page_label': '2'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024\nCEO 메시지\n회사 소개\n이해관계자 소통\nOur Company\n04\n05\n06\n준법과 윤리경영\nPrinciple\n53\n중대성 평가\nMateriality Assessment\n08\n임직원\n공급망\n사회공헌\n개인정보보호/보안\n고객의 안전/품질\nPeople\n31\n39\n45\n48\n50\n경제성과\n사회성과\n환경성과\n지역별 수자원 현황

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
len(chunks)

207

In [4]:
# 벡터스토어 및 리트리버 구성
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

embeddings = OpenAIEmbeddings()
persist_directory="../7_vectorstore/rag_eval_20"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="samsung2024_eval",
)

In [5]:
# 4. retriever 구성하기
retriever = vectorstore.as_retriever(
    search_kwargs = {"k" : 5}
)

In [6]:
#프롬프트 구성하기
from langchain_core.prompts import ChatPromptTemplate

system_template = """
    "You are a helpful assistant. Answer strictly based on the provided context. "
    "If the answer is not in the context, say you don't know."
    "context : {context}"
"""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{question}"),
])

In [7]:
# 모델 구성하기
model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [8]:
# 아웃풋 파서
from langchain_core.output_parsers import StrOutputParser
outputparser = StrOutputParser()

In [9]:
# 8. 체인설정
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# 문서 합치는 함수
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# 체인만들기
rag_chain = (
    {"context" : RunnableLambda(lambda x : x["question"]) | retriever | format_docs,
     "question" :RunnablePassthrough()
     }
     | rag_prompt
     | model
     | outputparser
)

In [10]:
rag_chain.invoke({"question": "삼성전자는 무슨일을 해?"})

'삼성전자는 제품 안전성 확보, 제품 품질 향상, 강제노동 방지, 그리고 재생에너지 확대 및 기후변화 대응을 위한 다양한 활동을 하고 있습니다. \n\n구체적으로, 삼성전자는 국제공인 시험소를 운영하여 제품 안전성 평가와 인증을 실시하고, 제품과 부품의 이중 안전 설계를 도입하여 사고 예방에 힘쓰고 있습니다. 또한, 전 세계에서 발생하는 품질 데이터를 분석하여 품질 문제 발생 시 조기 경보 및 긴급 개선조치를 실시하며, 고객 만족을 위한 품질 인증제도와 협력회사 품질 관리 제도를 운영하고 있습니다.\n\n더불어, 강제노동 방지를 위해 제조사업장의 근로조건을 정기적으로 평가하고, 특히 이주 근로자의 권리 보호를 위한 다양한 활동을 진행하고 있습니다. \n\n또한, 국내외 정부 및 여러 기관과 협력하여 재생에너지 확대와 에너지 효율화, 온실가스 감축 등 친환경 활동에도 적극 참여하고 있습니다.'

In [27]:
import pandas as pd
csv_path = "../../data/rag_eval.csv"
df = pd.read_csv(csv_path)
df

,user_input,reference_contexts,reference,synthesizer_name
0,What Samsung Electronics say about CSR D and E...,['Principle\nPlanet\nPeople\nCEO 메시지\nMessage ...,"Samsung Electronics explains that in 2023, glo...",single_hop_specific_query_synthesizer
1,한종희 삼성전자 부회장 지속가능경영에 대해 알려줘,"[""있어서는 비제조 분야 및 리스크 분석에 따라 제조 분야 2차 협력회사로 \n근로...",한종희 삼성전자 부회장은 지속가능경영을 삼성전자가 나아가야 할 방향의 흔들리지 않는...,single_hop_specific_query_synthesizer
2,Could you explain the role of the Device eXper...,['Principle\nPlanet\nPeople\n회사소개\nAbout Us\n삼...,The Device eXperience (DX) division at Samsung...,single_hop_specific_query_synthesizer
3,삼성닷컴 은 이해관계자 소통에서 어떤 역할을 하나요?,['Facts & Figures \nPrinciple\nPlanet\nPeople\...,"삼성닷컴은 고객과의 소통 채널 중 하나로, 제품과 서비스 품질, 안전한 제품 사용,...",single_hop_specific_query_synthesizer
4,What is the role of the 글로벌 구매 통합관리 시스템(G-SRM)...,['· 주주·투자자 의견 수렴\n 임직원\n· \x07안전하고 건강한 근로환경\n·...,The 글로벌 구매 통합관리 시스템(G-SRM) is part of the effo...,single_hop_specific_query_synthesizer
...,...,...,...,...
95,How did the company perform in terms of 폐전자제품 ...,['<1-hop>\n\n사내 폐기물 저감 실천 \n 폐제품 수거 체계 운영 상세내용...,"In 2021, the company collected 55.9 만 톤 of 폐전자...",multi_hop_specific_query_synthesizer
96,How did Samsung Electronics engage with intern...,['<1-hop>\n\n마련하고 유관 사업 활동을 분석하였습니다. 이후 각 활동과 ...,"In March 2024, Samsung Electronics conducted a...",multi_hop_specific_query_synthesizer
97,How DX부문 achieve 플래티넘 certification for 폐기물 매립...,['<1-hop>\n\n사내 폐기물 저감 실천 \n 폐제품 수거 체계 운영 상세내용...,DX부문 achieved the 플래티넘 certification for 폐기물 매...,multi_hop_specific_query_synthesizer
98,How DX부문 manage product and manufacturing haza...,['<1-hop>\n\n제품 및 제조과정 우려물질 관리\n∙ 제품 내 우려물질 및 ...,"In 2022년, DX부문 strengthened compliance and man...",multi_hop_specific_query_synthesizer


- user_input: 질문
- reference_contexts: 예상 되는 답변을 만들기 위해 참고한 context
- reference: 예상되는 답변

--------------------------

- user_input: 질문
- retrieved_contexts: 검색한 자료 -> 리스트
- response: 실제 답변

In [12]:
# for 문으로 구현하기
answer = []
contexts = []
for question in df['user_input']:
    docs = retriever.invoke(question)

    ctx_list = []
    for doc in docs:
        ctx_list.append(doc.page_content)
    contexts.append(ctx_list)


In [13]:
for question in df['user_input']:
    ans = rag_chain.invoke({"question": question})
    answer.append(ans)

In [14]:
answer

["The provided context from the 삼성전자 지속가능경영보고서 2024 (Samsung Electronics Sustainability Management Report 2024) highlights Samsung Electronics' commitment to Corporate Social Responsibility (CSR) and Environmental, Social, and Governance (ESG) practices as follows:\n\n1. **CSR and ESG Integration**: Samsung Electronics emphasizes listening to the opinions of partner companies and their workers in processes such as selecting new partners, ESG assessments, and grievance handling procedures. They have established and operate grievance handling systems, including hotlines and cyber complaint channels, to address labor environment violations and human rights issues confidentially and effectively.\n\n2. **ESG Assessments and Incentives**: Samsung operates an integrated labor environment management process consisting of self-assessments, on-site inspections, and third-party verifications. The results influence comprehensive evaluations and policy improvements. Since 2023, Samsung has implemen

In [16]:
df2 = pd.DataFrame({"user_input": df['user_input'], "retrieved_contexts": contexts, "response": answer})
df2.to_csv("../../data/rag_eval_result.csv", index=False)

In [28]:
df["response"] = answer
df["retrieved_contexts"] = contexts
df.head()


,user_input,reference_contexts,reference,synthesizer_name,response,retrieved_contexts
0,What Samsung Electronics say about CSR D and E...,['Principle\nPlanet\nPeople\nCEO 메시지\nMessage ...,"Samsung Electronics explains that in 2023, glo...",single_hop_specific_query_synthesizer,The provided context from the 삼성전자 지속가능경영보고서 2...,[삼성전자 지속가능경영보고서 2024\n40\nOur Company Appendix...
1,한종희 삼성전자 부회장 지속가능경영에 대해 알려줘,"[""있어서는 비제조 분야 및 리스크 분석에 따라 제조 분야 2차 협력회사로 \n근로...",한종희 삼성전자 부회장은 지속가능경영을 삼성전자가 나아가야 할 방향의 흔들리지 않는...,single_hop_specific_query_synthesizer,제공된 문서 내에는 한종희 삼성전자 부회장의 지속가능경영에 관한 구체적인 언급이나 ...,[삼성전자 지속가능경영보고서 2024\n04\nOur Company Appendix...
2,Could you explain the role of the Device eXper...,['Principle\nPlanet\nPeople\n회사소개\nAbout Us\n삼...,The Device eXperience (DX) division at Samsung...,single_hop_specific_query_synthesizer,The Device eXperience (DX) division within Sam...,[삼성전자 지속가능경영보고서 2024\n05\nOur Company Appendix...
3,삼성닷컴 은 이해관계자 소통에서 어떤 역할을 하나요?,['Facts & Figures \nPrinciple\nPlanet\nPeople\...,"삼성닷컴은 고객과의 소통 채널 중 하나로, 제품과 서비스 품질, 안전한 제품 사용,...",single_hop_specific_query_synthesizer,제공된 문맥에는 삼성닷컴이 이해관계자 소통에서 어떤 역할을 하는지에 대한 정보가 없...,[마련하고 유관 사업 활동을 분석하였습니다. 이후 각 활동과 관련성이 높은 \n밸류...
4,What is the role of the 글로벌 구매 통합관리 시스템(G-SRM)...,['· 주주·투자자 의견 수렴\n 임직원\n· \x07안전하고 건강한 근로환경\n·...,The 글로벌 구매 통합관리 시스템(G-SRM) is part of the effo...,single_hop_specific_query_synthesizer,The provided context does not mention the 글로벌 ...,[완화하기 위한 개선 활동을 실시합니다. \n리스크 관리 \n정책 \n협력회사 행동...


In [30]:
import ast
def safe_literal_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return x
    return x

In [31]:

df["retrieved_contexts"] = df["retrieved_contexts"].apply(lambda x: safe_literal_eval(x))
df["reference_contexts"] = df["reference_contexts"].apply(lambda x: safe_literal_eval(x))

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   user_input          100 non-null    object
 1   reference_contexts  100 non-null    object
 2   reference           100 non-null    object
 3   synthesizer_name    100 non-null    object
 4   response            100 non-null    object
 5   retrieved_contexts  100 non-null    object
dtypes: object(6)
memory usage: 4.8+ KB


In [ ]:
df.info()

In [32]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness, LLMContextRecall, FactualCorrectness
from ragas.llms import LangchainLLMWrapper

# ragas 평가
eval_llm = LangchainLLMWrapper(model)
dataset = EvaluationDataset.from_pandas(df)
dataset

C:\Users\user\AppData\Local\Temp\ipykernel_20472\3001053187.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  eval_llm = LangchainLLMWrapper(model)


EvaluationDataset(features=['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference'], len=100)

In [ ]:
scores = evaluate(
    dataset,
    metrics=[
        Faithfulness(), # 높으면 할루시네이션 없는거
        LLMContextRecall(), # 낮으면 문서를 제대로 찾아오지 못함 높으면 잘 찾아옴
        FactualCorrectness(), # 낮으면 프롬프트 형식에 문제가 있음
    ],
    llm=eval_llm
)

Evaluating:   0%|          | 0/300 [00:00<?, ?it/s]

Exception raised in Job[27]: TimeoutError()
Exception raised in Job[31]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[30]: TimeoutError()
Exception raised in Job[34]: TimeoutError()
Exception raised in Job[33]: TimeoutError()
Exception raised in Job[32]: TimeoutError()
Exception raised in Job[35]: TimeoutError()
Exception in thread Thread-155:
Traceback (most recent call last):
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.11.13-windows-x86_64-none\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\user\potenup\python7month\LangChainProject\.venv\Lib\site-packages\tqdm\_monitor.py", line 84, in run
    instance.refresh(nolock=True)
  File "c:\Users\user\potenup\python7month\LangChainProject\.venv\Lib\site-packages\tqdm\std.py", line 1347, in refresh
    self.display()
  File "c:\Users\user\potenup\python7month\LangChainProject\.venv\Lib\site-packages\tqdm\notebook.py", line 157, in display
    pbar.value =

KeyboardInterrupt: 

: 

In [ ]:
scores